In [0]:
%python
from pyspark.sql.functions import(
    col,
    lit,
    trim,
    lower,
    upper,
    to_timestamp,
    when
)

In [0]:
%python
customers = spark.table("ecommerce_lakehouse.bronze.customers")

In [0]:
%python
silver_customers = (
    customers.select(
        trim(col("customer_id")).alias("customer_id"),
        trim(col("customer_unique_id")).alias("customer_unique_id"),
        col("customer_zip_code_prefix").alias("customer_zip_code_prefix"),
        upper(trim(col("customer_city"))).alias("customer_city"),
        upper(trim(col("customer_state"))).alias("customer_state"),
        )
    .dropDuplicates(["customer_id"])
)

(
    silver_customers
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_lakehouse.silver.customers")
)

In [0]:
%python
orders = spark.table("ecommerce_lakehouse.bronze.orders")

In [0]:
%python
silver_orders = (
    orders
    .select(
        trim(col("order_id")).alias("order_id"),
        trim(col("customer_id")).alias("customer_id"),
        lower(trim(col("order_status"))).alias("order_status"),

        to_timestamp(
            col("order_purchase_timestamp")
        ).alias("order_purchase_timestamp"),

        to_timestamp(
            col("order_approved_at")
        ).alias("order_approved_at"),

        to_timestamp(
            col("order_delivered_carrier_date")
        ).alias("order_delivered_carrier_date"),

        to_timestamp(
            col("order_delivered_customer_date")
        ).alias("order_delivered_customer_date"),

        to_timestamp(
            col("order_estimated_delivery_date")
        ).alias("order_estimated_delivery_date")
    )
    .dropDuplicates(["order_id"])
)

(
    silver_orders
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_lakehouse.silver.orders")
)

In [0]:
%python
order_items = spark.table("ecommerce_lakehouse.bronze.order_items")

In [0]:
%python
silver_order_items = (
    order_items
    .select(
        trim(col("order_id")).alias("order_id"),
        trim(col("order_item_id")).alias("order_item_id"),
        trim(col("product_id")).alias("product_id"),
        trim(col("seller_id")).alias("seller_id"),
        to_timestamp(
            col("shipping_limit_date")
        ).alias("shipping_limit_date"),
        col("price").alias("price"),
        col("freight_value").alias("freight_value"),

    )
    .dropDuplicates([
        "order_id",
        "order_item_id"
    ])
)

(
    silver_order_items
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_lakehouse.silver.order_items")
)

In [0]:
%python
products = spark.table("ecommerce_lakehouse.bronze.products")

In [0]:
%python
silver_products = (
    products
    .select(
        trim(col("product_id")).alias("product_id"),
        lower(trim(col("product_category_name"))).alias("product_category_name"),
        col("product_name_lenght").alias("product_name_length"),
        col("product_description_lenght").alias("product_description_length"),
        col("product_photos_qty").alias("product_photos_qty"),
        col("product_weight_g").alias("product_weight_g"),
        col("product_length_cm").alias("product_length_cm"),
        col("product_height_cm").alias("product_height_cm"),
        col("product_width_cm").alias("product_width_cm"),
    )
    .dropDuplicates(["product_id"])
)

(
    silver_products
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_lakehouse.silver.products")
)

In [0]:
%python
sellers = spark.table("ecommerce_lakehouse.bronze.sellers")

In [0]:
%python
silver_sellers = (
    sellers
    .select(
        trim(col("seller_id")).alias("seller_id"),
        col("seller_zip_code_prefix").alias("seller_zip_code_prefix"),
        upper(trim(col("seller_city"))).alias("seller_city"),
        upper(trim(col("seller_state"))).alias("seller_state")
    )
    .dropDuplicates(["seller_id"])
)
(
    silver_sellers
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_lakehouse.silver.sellers")
)

In [0]:
%python
payments = spark.table("ecommerce_lakehouse.bronze.payments")

In [0]:
%python
silver_payments = (
    payments
    .select(
        trim(col("order_id")).alias("order_id"),
        col("payment_sequential").alias("payment_sequential"),
        lower(trim(col("payment_type"))).alias("payment_type"),
        col("payment_installments").alias("payment_installments"),
        col("payment_value").alias("payment_value")
    )
)

(
    silver_payments
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_lakehouse.silver.payments")
)

In [0]:
%python
reviews = spark.table("ecommerce_lakehouse.bronze.reviews")

In [0]:
%python
from pyspark.sql.functions import expr

silver_reviews = (
    reviews
    .select(
        trim(col("review_id")).alias("review_id"),
        trim(col("order_id")).alias("order_id"),
        expr("try_cast(review_score AS INT)").alias("review_score"),

        trim(col("review_comment_title"))
            .alias("review_comment_title"),

        trim(col("review_comment_message"))
            .alias("review_comment_message"),

        expr("try_to_timestamp(review_creation_date)").alias("review_creation_date"),

        expr("try_to_timestamp(review_answer_timestamp)").alias("review_answer_timestamp")
    )
    .dropDuplicates(["review_id"])
)

(
    silver_reviews
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_lakehouse.silver.reviews")
)

In [0]:
%python
categories = spark.table("ecommerce_lakehouse.bronze.category_translation")

In [0]:
%python
silver_categories = (
    categories
    .select(
        lower(trim(col("product_category_name")))
            .alias("product_category_name"),

        lower(trim(col("product_category_name_english")))
            .alias("product_category_name_english")
    )
    .dropDuplicates(["product_category_name"])
)
(
    silver_categories
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ecommerce_lakehouse.silver.category_translation"
    )
)

In [0]:
%python
geolocation = spark.table("ecommerce_lakehouse.bronze.geolocations")

In [0]:
%python
silver_geolocation = (
    geolocation
    .select(
        col("geolocation_zip_code_prefix")
            .cast("int")
            .alias("zip_code_prefix"),

        col("geolocation_lat")
            .cast("double")
            .alias("latitude"),

        col("geolocation_lng")
            .cast("double")
            .alias("longitude"),

        upper(trim(col("geolocation_city")))
            .alias("city"),

        upper(trim(col("geolocation_state")))
            .alias("state")
    )
)



from pyspark.sql.functions import avg, first

silver_geolocation = (
    silver_geolocation
    .groupBy("zip_code_prefix")
    .agg(
        avg("latitude").alias("latitude"),
        avg("longitude").alias("longitude"),
        first("city", ignorenulls=True).alias("city"),
        first("state", ignorenulls=True).alias("state")
    )
)

(
    silver_geolocation
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ecommerce_lakehouse.silver.geolocation"
    )
)

In [0]:
SHOW TABLES IN ecommerce_lakehouse.silver;

database,tableName,isTemporary
silver,category_translation,false
silver,customers,false
silver,geolocation,false
silver,order_items,false
silver,orders,false
silver,payments,false
silver,products,false
silver,reviews,false
silver,sellers,false
